In [17]:
import sys
import os

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)


In [1]:
import json
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    OneHotEncoder, PowerTransformer
)
from sklearn.impute import SimpleImputer


In [3]:
df = pd.read_csv("../data/titanic.csv")

with open("../outputs/preprocessing_plan.json") as f:
    preprocessing_plan = json.load(f)

target_col = "Survived"
X = df.drop(columns=[target_col])
y = df[target_col]


In [4]:
X = X.drop(columns=preprocessing_plan["drop_columns"])


In [5]:
numerical_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(exclude=np.number).columns.tolist()


In [6]:
num_imputer = SimpleImputer(strategy="median")


In [7]:
cat_imputer = SimpleImputer(strategy="most_frequent")


In [8]:
categorical_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)


In [9]:
apply_power = any(
    t["method"] == "power" for t in preprocessing_plan["transformation"]
)


In [10]:
power_transformer = PowerTransformer() if apply_power else "passthrough"


In [18]:
from src.pipelines import build_pipelines

pipelines = build_pipelines(X, preprocessing_plan)

pipelines.keys()


dict_keys(['standard', 'minmax', 'robust'])

In [14]:
X_transformed = pipelines["standard"].fit_transform(X)
X_transformed.shape


(891, 1583)

In [15]:
pipeline_metadata = {
    "pipelines_generated": list(pipelines.keys()),
    "common_steps": ["imputation", "encoding", "scaling", "transformation"]
}

with open("../outputs/pipeline_metadata.json", "w") as f:
    json.dump(pipeline_metadata, f, indent=4)


Tadaa constructs multiple preprocessing pipelines automatically using statistically-driven decisions, enabling fair comparison without manual configuration.